# 02 — Data Cleaning & Feature Engineering
Transforms raw match data into a model-ready dataset: results, points, gameweek, title gap, high-stakes flags, Drop Index, rolling features, and recency weights.

## Imports

In [1]:
import pandas as pd
import numpy as np
import soccerdata as sd
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")
print("\n✅ Imports ready")

[08/31/26 23:02:19] INFO     No custom team name replacements found. You can configure these in       ]8;id=6696185;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=6696186;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py#91\91]8;;\
                             C:\Users\tejas\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=6696192;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=6696193;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py#189\189]8;;\
                             C:\Users\tejas\soccerdata\config\league_dict.json.                                    

pandas : 3.0.5
numpy  : 2.4.6

✅ Imports ready


## Paths & Config

In [2]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROC_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROC_DATA_DIR.mkdir(parents=True, exist_ok = True)

TITLE_TEAMS = ['Arsenal','Liverpool','Manchester City','Manchester United']
PL = "ENG-Premier League"
ARTETA_SEASONS = ['1920', '2021', '2122', '2223', '2324', '2425', '2526']

# Recency weights per season — 25-26 counts 1.5x in the ML model
SEASON_WEIGHTS = {
    '1920': 1.0, '2021': 1.0, '2122': 1.0,
    '2223': 1.1, '2324': 1.2, '2425': 1.3,
    '2526': 1.5,
}
# Threshold for binary high/low stakes label (Drop Index bucket comparison)
STAKES_THRESHOLD = 0.4

# Rivalry matches — always get a flat +0.15 stakes bonus
RIVALRIES = {
    'Arsenal' : ['Manchester City', 'Chelsea', 'Tottenham'],
    'Liverpool' : ['Manchester United', 'Manchester City', 'Everton'],
    'Manchester City' : ['Manchester United', 'Liverpool', 'Arsenal'],
    'Manchester United': ['Liverpool', 'Manchester City', 'Arsenal', 'Chelsea']
}

print(f"Raw data dir       : {RAW_DATA_DIR}")
print(f"Processed data dir : {PROC_DATA_DIR}")
print(f"Season weights     : {SEASON_WEIGHTS}")
print(f"Stakes threshold   : {STAKES_THRESHOLD}")

Raw data dir       : C:\Users\tejas\OneDrive\Desktop\Arsenal Bottle Python\data\raw
Processed data dir : C:\Users\tejas\OneDrive\Desktop\Arsenal Bottle Python\data\processed
Season weights     : {'1920': 1.0, '2021': 1.0, '2122': 1.0, '2223': 1.1, '2324': 1.2, '2425': 1.3, '2526': 1.5}
Stakes threshold   : 0.4


## Load and Combine All 4 Teams

In [3]:
# Load all 4 title teams and stack into one DataFrame

dfs = [
    pd.read_csv(RAW_DATA_DIR / f"{team.lower().replace(' ', '_')}_raw.csv")
    for team in TITLE_TEAMS
]

df = pd.concat(dfs, ignore_index = True)

# Fix date column type
df['date'] = pd.to_datetime(df['date'])

# Sort by team then date — critical for rolling features to work correctly
df = df.sort_values(['team','date']).reset_index(drop=True)


print(f"Combined shape : {df.shape}")
print(f"Teams          : {sorted(df['team'].unique())}")
print(f"Date range     : {df['date'].min().date()} → {df['date'].max().date()}")
df.head(5)

Combined shape : (1064, 11)
Teams          : ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United']
Date range     : 2019-08-09 → 2026-05-24


,league,season,game_id,date,team,opponent,venue,xG,xGA,scored,conceded
0,ENG-Premier League,1920,11650,2019-08-11 14:00:00,Arsenal,Newcastle United,away,1.133090,0.380551,1,0
1,ENG-Premier League,1920,11653,2019-08-17 12:30:00,Arsenal,Burnley,home,1.164400,1.391720,2,1
2,ENG-Premier League,1920,11670,2019-08-24 17:30:00,Arsenal,Liverpool,away,0.985542,2.788210,1,3
3,ENG-Premier League,1920,11682,2019-09-01 16:30:00,Arsenal,Tottenham,home,1.925090,1.955140,2,2
4,ENG-Premier League,1920,11691,2019-09-15 15:30:00,Arsenal,Watford,away,1.006160,2.832090,2,2


## Compute Result and Points

In [4]:
# Compute result — W/D/L from scored vs conceded

df['result'] = np.where(
    df['scored'] > df['conceded'], 'W',
    np.where(df['scored'] == df['conceded'], 'D', 'L')
)

# Compute points — 3 for win, 1 for draw, 0 for loss
df['points'] = df['result'].map({'W' : 3, 'D' : 1, 'L' : 0})

# Compute goal difference per match
df['gd'] = df['scored'] - df['conceded']

# Compute xG difference
df['xgd'] = df['xG'] - df['xGA']

print("Result distribution across all 4 teams:")
print(df['result'].value_counts())
print()
print("Points distribution:")
print(df['points'].value_counts().sort_index())
print()
print("Spot check — first 5 Arsenal rows:")
df[df['team'] == 'Arsenal'][['date', 'opponent', 'scored', 'conceded', 'result', 'points']].head(5)

Result distribution across all 4 teams:
result
W    628
D    222
L    214
Name: count, dtype: int64

Points distribution:
points
0    214
1    222
3    628
Name: count, dtype: int64

Spot check — first 5 Arsenal rows:


,date,opponent,scored,conceded,result,points
0,2019-08-11 14:00:00,Newcastle United,1,0,W,3
1,2019-08-17 12:30:00,Burnley,2,1,W,3
2,2019-08-24 17:30:00,Liverpool,1,3,L,0
3,2019-09-01 16:30:00,Tottenham,2,2,D,1
4,2019-09-15 15:30:00,Watford,2,2,D,1


## Add Gameweek

In [5]:
# Compute gameweek: rank matches chronologically within each team-season

df['gameweek'] = (
    df.groupby(['team','season'])['date']
    .rank(method = 'dense')
    .astype(int)
)

# Verify — each team in each season should have gameweeks 1 to 38
gw_check = df.groupby(['team', 'season'])['gameweek'].max().unstack(fill_value=0)
print("Max gameweek per team per season (should all be 38):")
print(gw_check)

Max gameweek per team per season (should all be 38):
season             1920  2021  2122  2223  2324  2425  2526
team                                                       
Arsenal              38    38    38    38    38    38    38
Liverpool            38    38    38    38    38    38    38
Manchester City      38    38    38    38    38    38    38
Manchester United    38    38    38    38    38    38    38


## Reconstruct Full PL Title Table

In [6]:
print("Loading full 20-team schedule from cache...")

understat = sd.Understat(leagues=PL, seasons=ARTETA_SEASONS)
full_schedule = understat.read_schedule().reset_index()

print(f"Shape: {full_schedule.shape}")
print("Done — loaded from local cache")

Loading full 20-team schedule from cache...


                    INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=6696200;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=6696201;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-08-31 23:02:19] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Successfully loaded TLS library:                                      ]8;id=6696208;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=6696209;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\                 
                             bin\tls-client-xgo-1.13.1-windows-amd64.dll                                           

Shape: (2660, 20)
Done — loaded from local cache


In [7]:
KEEP = ['league', 'season', 'game_id', 'date', 'team', 'scored', 'conceded']

full_home = full_schedule.rename(columns={
    'home_team': 'team', 'home_goals': 'scored', 'away_goals': 'conceded'
})
full_away = full_schedule.rename(columns={
    'away_team': 'team', 'away_goals': 'scored', 'home_goals': 'conceded'
})

full_long = pd.concat([full_home[KEEP],full_away[KEEP]],ignore_index=True)
full_long['date'] = pd.to_datetime(full_long['date'])

# Compute points for each team
full_long['points'] = np.where(
    full_long['scored'] > full_long['conceded'],3,
    np.where(full_long['scored'] == full_long['conceded'],1,0)
)

# Sort chronologically within each team-season before cumsum
full_long = full_long.sort_values(['team','season','date']).reset_index(drop = True)

print(f"Full 20-team long format: {full_long.shape}")


Full 20-team long format: (5320, 8)


In [8]:
#Compute cumsum

full_long['cum_pts'] = full_long.groupby(['team','season'])['points'].cumsum()

full_long['gw'] = (
    full_long.groupby(['team','season'])['date']
    .rank(method = 'dense')
    .astype(int)
)

print("Arsenal cumulative points (first 5 GW of 2019-20):")
mask = (full_long['team'] == 'Arsenal') & (full_long['season'] == '1920')
print(full_long[mask][['date', 'gw', 'points', 'cum_pts']].head(5))

Arsenal cumulative points (first 5 GW of 2019-20):
                 date  gw  points  cum_pts
0 2019-08-11 14:00:00   1       3        3
1 2019-08-17 12:30:00   2       3        6
2 2019-08-24 17:30:00   3       0        6
3 2019-09-01 16:30:00   4       1        7
4 2019-09-15 15:30:00   5       1        8


## Compute Title Gap and All Positional Boundaries

In [9]:
# ------------------ TITLE RACE -----------------------
full_long['leader_pts'] = (
    full_long.groupby(['season','gw'])['cum_pts'].transform('max')
)
full_long['title_gap'] = full_long['leader_pts'] - full_long['cum_pts']

# ---------CHAMPIONS LEAGUE BOUNDARY(4th/5th)----------
full_long['pts_4th'] = (
    full_long.groupby(['season','gw'])['cum_pts']
    .transform(lambda x : x.nlargest(4).iloc[-1])
)
full_long['pts_5th'] = (
    full_long.groupby(['season','gw'])['cum_pts']
    .transform(lambda x : x.nlargest(5).iloc[-1])
)

# ------------ EUROPA BOUNDARY (6th/7th) ------------
full_long['pts_6th'] = (
    full_long.groupby(['season','gw'])['cum_pts']
    .transform(lambda x : x.nlargest(6).iloc[-1])
)
full_long['pts_7th'] = (
    full_long.groupby(['season','gw'])['cum_pts']
    .transform(lambda x : x.nlargest(7).iloc[-1])
)

# SYMMETRIC GAP TO EACH BOUNDARY
# If IN the zone -> gap = how close is the team just outside
# If OUT the zone -> gap = how far is team from getting in

def boundargy_gap(cum_pts,pts_top,pts_next):
    return np.where(
        cum_pts >= pts_top,
        pts_top - pts_next,
        pts_top - cum_pts
    )

full_long['cl_gap'] = boundargy_gap(
    full_long['cum_pts'], full_long['pts_4th'],full_long['pts_5th']
)
full_long['eur_gap'] = boundargy_gap(
    full_long['cum_pts'], full_long['pts_6th'],full_long['pts_7th']
)

# Spot check  GW30 2019-20: Liverpool dominant, Arsenal should have large gaps
mask = (full_long['season'] == '1920') & (full_long['gw'] == 30)
print("GW30 2019-20 — top 6 teams with all gaps:")
print(
    full_long[mask][['team', 'cum_pts', 'title_gap', 'cl_gap', 'eur_gap']]
    .sort_values('title_gap')
    .head(10)
)

GW30 2019-20 — top 6 teams with all gaps:
                         team  cum_pts  title_gap  cl_gap  eur_gap
2765                Liverpool       83          0       5        2
3069          Manchester City       63         20       5        2
2575                Leicester       54         29       5        2
1397                  Chelsea       51         32       5        2
3335        Manchester United       46         37       5        2
5083  Wolverhampton Wanderers       46         37       5        2
4095         Sheffield United       44         39       7        2
1663           Crystal Palace       42         41       9        4
4437                Tottenham       42         41       9        4
29                    Arsenal       40         43      11        6


In [10]:
#Extracting our 4 teams with all gap columns
title_gap_df = (
    full_long[full_long['team'].isin(TITLE_TEAMS)]
    [['season','team','gw','cum_pts','leader_pts','title_gap','cl_gap','eur_gap']]
    .copy()
    .rename(columns={'gw' : 'gameweek'})
)

df['season'] = df['season'].astype(str)

print(f"Title gap table shape: {title_gap_df.shape}")
print(f"Expected: 4 × 7 × 38 = {4*7*38} rows")
title_gap_df.head(5)

Title gap table shape: (1064, 8)
Expected: 4 × 7 × 38 = 1064 rows


,season,team,gameweek,cum_pts,leader_pts,title_gap,cl_gap,eur_gap
0,1920,Arsenal,1,3,3,0,0,0
1,1920,Arsenal,2,6,6,0,0,0
2,1920,Arsenal,3,6,9,3,1,0
3,1920,Arsenal,4,7,12,5,0,0
4,1920,Arsenal,5,8,15,7,0,0


## Merge Title Gap into Main DataFrame

In [11]:
# Merge title_gap into our main df

df = pd.merge(
    df,
    title_gap_df,
    on=['season', 'team', 'gameweek'],
    how='left'
)


print(f"Shape after merge: {df.shape}")
print(f"title_gap nulls  : {df['title_gap'].isna().sum()}")
print(f"cl_gap nulls    : {df['cl_gap'].isna().sum()}")
print(f"eur_gap nulls    : {df['eur_gap'].isna().sum()}")

# Spot check — Arsenal 2021-22, final gameweek
mask = (df['team'] == 'Liverpool') & (df['season'] == '2122') & (df['gameweek'] >= 36)
df[mask][['date', 'gameweek', 'opponent', 'result', 'cum_pts', 'title_gap', 'cl_gap','eur_gap']].sort_values('gameweek')

Shape after merge: (1064, 21)
title_gap nulls  : 0
cl_gap nulls    : 0
eur_gap nulls    : 0


,date,gameweek,opponent,result,cum_pts,title_gap,cl_gap,eur_gap
377,2022-05-10 19:00:00,36,Aston Villa,W,86,3,1,3
378,2022-05-17 18:45:00,37,Southampton,W,89,1,2,2
379,2022-05-22 15:00:00,38,Wolverhampton Wanderers,W,92,1,2,2


## Define Stakes Intensity

In [12]:
# Normalization constant
# Maximum raw valeu: GW38, title_gap=0 
# Dividing by this makes the ceiling = 1.0
GW38_MAX = 1 / (1 + np.exp(-0.2 * (38-22)))

# Time Pressure (same for all components)
gw_factor = 1 / (1 + np.exp(-0.2 * (df['gameweek'] - 22)))

# Points remaining (relative gap denominator)
# At GW38: 3pts. At GW1: 114 pts
pts_remaining = (38 - df['gameweek'] + 1) * 3

# Individual gap factors
title_f = (1 - df['title_gap'] / pts_remaining).clip(0)
cl_f = (1 - df['cl_gap'] / pts_remaining).clip(0)
eur_f = (1 - df['eur_gap'] / pts_remaining).clip(0)

# Raw component scores
title_raw = gw_factor * title_f       #caps at 1.00
cl_raw = gw_factor * cl_f * 0.75      #caps at 0.75
eur_raw = gw_factor * eur_f * 0.50    #caps at 0.5

# Take the maximum
base_stakes = np.maximum.reduce([title_raw, cl_raw, eur_raw]) / GW38_MAX

# Form pressure multiplier
# If a team is on poor form (avg<1 pt/game last 3), urgency is higher
recent_form = (
    df.groupby(['team','season'])['points']
    .transform(lambda x : x.shift(1).rolling(3,min_periods = 1).mean())
    .fillna(1.5)
)
form_multiplier = np.where(recent_form < 1.0, 1.15, 1.0)

# Rivalry bonus
# Flat +0.15 for derbies and historical rivals
df['is_rivalry'] = df.apply(
    lambda row: row['opponent'] in RIVALRIES.get(row['team'],[]),
    axis = 1
)
rivalry_bonus = np.where(df['is_rivalry'],0.15,0)

# FINAL STAKES INTENSITY
df['stakes_intensity'] = (
    (base_stakes * form_multiplier + rivalry_bonus)
    .clip(0, 1)
    .round(4)
)

print("stakes_intensity distribution:")
print(df['stakes_intensity'].describe().round(3))
print()
print("Mean stakes_intensity per team:")
print(df.groupby('team')['stakes_intensity'].mean().round(3))
print()
ars = df[df['team'] == 'Arsenal'].sort_values('stakes_intensity', ascending=False)
print("Arsenal top 5 highest-stakes matches:")
print(ars[['date', 'season', 'gameweek', 'opponent', 'title_gap', 'cl_gap', 'is_rivalry', 'stakes_intensity']].head(5).to_string())

stakes_intensity distribution:
count    1064.000
mean        0.367
std         0.288
min         0.000
25%         0.110
50%         0.320
75%         0.572
max         1.000
Name: stakes_intensity, dtype: float64

Mean stakes_intensity per team:
team
Arsenal              0.356
Liverpool            0.391
Manchester City      0.421
Manchester United    0.303
Name: stakes_intensity, dtype: float64

Arsenal top 5 highest-stakes matches:
                   date season  gameweek         opponent  title_gap  cl_gap  is_rivalry  stakes_intensity
265 2026-05-24 15:00:00   2526        38   Crystal Palace          0       5       False            1.0000
260 2026-04-19 15:30:00   2526        33  Manchester City          0       3        True            1.0000
264 2026-05-18 20:00:00   2526        37          Burnley          0       3       False            0.9914
256 2026-03-01 16:30:00   2526        29          Chelsea          0       3        True            0.9849
263 2026-05-10 16:30:00   2

In [15]:
# Season-wise distribution — diagnostic to verify stakes are spread correctly
season_dist = (
    df.groupby(['team', 'season', 'is_high_stakes'])
    .agg(
        n          = ('xG',              'count'),
        avg_xG     = ('xG',              'mean'),
        avg_xGA    = ('xGA',             'mean'),
        avg_stakes = ('stakes_intensity','mean'),
        avg_pts    = ('points',          'mean'),
    )
    .round(3)
)
# Note: is_high_stakes not yet defined — run this cell AFTER Step 10a
print("Run this cell after Step 10a once is_high_stakes is defined.")

Run this cell after Step 10a once is_high_stakes is defined.


## Build the Drop Index

In [51]:
#Binary label for Drop Index bucket comparison
# The ML Model will always use the continuous stakes_intensity, never this
df['is_high_stakes'] = df['stakes_intensity'] >= STAKES_THRESHOLD

print(f"High-stakes matches (stakes >= {STAKES_THRESHOLD}) per team:")
print(df.groupby('team')['is_high_stakes'].sum())

High-stakes matches (stakes >= 0.4) per team:
team
Arsenal              104
Liverpool            123
Manchester City      122
Manchester United     98
Name: is_high_stakes, dtype: int64


In [52]:
# Contention season filter
# Only keep team-seasons with >= 5 high-stakes matches
# Without this, 'normal' bucket = bad seasons, 'high stakes' = good seasons
# = comparing good Arsenal to bad Arsenal, not pressure vs no pressure

bucket_counts = (
    df.groupby(['team','season','is_high_stakes'])
    .size()
    .unstack(fill_value=0)
    .rename(columns = {False:'n_normal', True: 'n_highstakes'})
)

contention = bucket_counts[bucket_counts['n_highstakes'] >= 5]
print(contention)

df_drop = df.merge(
    contention.reset_index()[['team','season']],
    on=['team','season'],
    how='inner'
)

print(f"\nRows used for Drop Index: {len(df_drop)} / {len(df)} total")
print(f"Teams included: {sorted(df_drop['team'].unique())}")

is_high_stakes            n_normal  n_highstakes
team              season                        
Arsenal           1920          29             9
                  2021          30             8
                  2122          22            16
                  2223          20            18
                  2324          21            17
                  2425          21            17
                  2526          19            19
Liverpool         1920          19            19
                  2021          18            20
                  2122          20            18
                  2223          24            14
                  2324          20            18
                  2425          18            20
                  2526          24            14
Manchester City   1920          23            15
                  2021          19            19
                  2122          19            19
                  2223          19            19
                  23

In [53]:
# RAW DROP INDEX

perf = (
    df_drop.groupby(['team','is_high_stakes'])
    .agg(
        xG_mean = ('xG','mean'),
        xGA_mean = ('xGA','mean'),
        n_matches = ('xG','count')
    ).round(3)
)

print("Mean xG and xGA: normal (False) vs high-stakes (True):")
print(perf)

Mean xG and xGA: normal (False) vs high-stakes (True):
                                  xG_mean  xGA_mean  n_matches
team              is_high_stakes                              
Arsenal           False             1.793     1.090        162
                  True              1.808     1.176        104
Liverpool         False             2.171     1.218        143
                  True              2.163     1.178        123
Manchester City   False             2.273     1.043        144
                  True              2.232     0.931        122
Manchester United False             1.745     1.351        133
                  True              1.673     1.395         95


In [54]:
perf_wide = perf[['xG_mean', 'xGA_mean']].unstack('is_high_stakes')

contention_teams = sorted(df_drop['team'].unique())
drop_index = pd.DataFrame(index=contention_teams)

drop_index['xG_normal']      = perf_wide[('xG_mean',  False)]
drop_index['xG_highstakes']  = perf_wide[('xG_mean',  True)]
drop_index['xGA_normal']     = perf_wide[('xGA_mean', False)]
drop_index['xGA_highstakes'] = perf_wide[('xGA_mean', True)]

# Positive = performs worse under pressure (the bottle)
drop_index['drop_xG']    = drop_index['xG_normal']      - drop_index['xG_highstakes']
drop_index['drop_xGA']   = drop_index['xGA_highstakes'] - drop_index['xGA_normal']
drop_index['Drop_Index'] = (drop_index['drop_xG'] + drop_index['drop_xGA']) / 2

print("RAW Drop Index (xG units) — contention seasons only:")
print(drop_index[['drop_xG', 'drop_xGA', 'Drop_Index']].round(3).sort_values('Drop_Index', ascending=False))

RAW Drop Index (xG units) — contention seasons only:
                   drop_xG  drop_xGA  Drop_Index
Manchester United    0.072     0.044       0.058
Arsenal             -0.015     0.086       0.035
Liverpool            0.008    -0.040      -0.016
Manchester City      0.041    -0.112      -0.035


In [55]:
# ── Z-SCORE Drop Index ────────────────────────────────────────────────────────
# Standardize within each team: z = (xG - team_mean) / team_std
# Makes cross-team comparison fair regardless of different baseline xG levels

df['xG_z']  = df.groupby('team')['xG'].transform(
    lambda x: (x - x.mean()) / x.std()
)
df['xGA_z'] = df.groupby('team')['xGA'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# Recompute df_drop to include xG_z and xGA_z
df_drop = df.merge(
    contention.reset_index()[['team', 'season']],
    on=['team', 'season'],
    how='inner'
)

perf_z = (
    df_drop.groupby(['team', 'is_high_stakes'])
    .agg(
        xG_z_mean  = ('xG_z',  'mean'),
        xGA_z_mean = ('xGA_z', 'mean'),
    )
)

perf_z_wide = perf_z.unstack('is_high_stakes')

drop_index['xG_z_normal']      = perf_z_wide[('xG_z_mean',  False)]
drop_index['xG_z_highstakes']  = perf_z_wide[('xG_z_mean',  True)]
drop_index['xGA_z_normal']     = perf_z_wide[('xGA_z_mean', False)]
drop_index['xGA_z_highstakes'] = perf_z_wide[('xGA_z_mean', True)]

drop_index['drop_xG_z']    = drop_index['xG_z_normal']      - drop_index['xG_z_highstakes']
drop_index['drop_xGA_z']   = drop_index['xGA_z_highstakes'] - drop_index['xGA_z_normal']
drop_index['Drop_Index_z'] = (drop_index['drop_xG_z'] + drop_index['drop_xGA_z']) / 2

print("Z-SCORE Drop Index (standard deviation units) — contention seasons only:")
print(drop_index[['drop_xG_z', 'drop_xGA_z', 'Drop_Index_z']].round(3).sort_values('Drop_Index_z', ascending=False))
print()
print("Interpretation: Drop_Index_z = 0.3 means the team performs 0.3 SD below")
print("their own average in high-stakes matches (combined attack + defence)")

Z-SCORE Drop Index (standard deviation units) — contention seasons only:
                   drop_xG_z  drop_xGA_z  Drop_Index_z
Manchester United      0.077       0.050         0.064
Arsenal               -0.015       0.103         0.044
Liverpool              0.007      -0.052        -0.022
Manchester City        0.041      -0.147        -0.053

Interpretation: Drop_Index_z = 0.3 means the team performs 0.3 SD below
their own average in high-stakes matches (combined attack + defence)


## Rolling Features
**Critical:** always `.shift(1)` before `.rolling()` — no exceptions.

In [61]:
def roll(series_grouped, window, min_p=3):
    """Lag-1 rolling mean: shift first, then roll. Prevents leakage."""
    return series_grouped.transform(
        lambda x: x.shift(1).rolling(window, min_periods=min_p).mean()
    )

g = df.groupby(['team', 'season'])

df['xG_roll5']        = roll(g['xG'],     5)
df['xG_roll10']       = roll(g['xG'],    10)
df['xGA_roll5']       = roll(g['xGA'],    5)
df['xGA_roll10']      = roll(g['xGA'],   10)
df['pts_roll5']       = roll(g['points'], 5)
df['pts_roll10']      = roll(g['points'],10)
df['is_win']          = (df['result'] == 'W').astype(int)
df['win_rate_roll5']  = roll(g['is_win'], 5)
df['win_rate_roll10'] = roll(g['is_win'],10)

rolling_cols = ['xG_roll5','xG_roll10','xGA_roll5','xGA_roll10',
                'pts_roll5','pts_roll10','win_rate_roll5','win_rate_roll10']

print("Rolling feature null counts (early-season NaN is expected):")
print(df[rolling_cols].isna().sum())

Rolling feature null counts (early-season NaN is expected):
xG_roll5           84
xG_roll10          84
xGA_roll5          84
xGA_roll10         84
pts_roll5          84
pts_roll10         84
win_rate_roll5     84
win_rate_roll10    84
dtype: int64


In [62]:
# Leakage verification: GW1 must have NaN rolling features (no prior history)
arsenal_1920 = df[(df['team']=='Arsenal') & (df['season']=='1920')]
print("Arsenal 2019-20 first 6 matches — xG_roll5 must lag by 1:")
print(arsenal_1920[['date','gameweek','xG','xG_roll5','pts_roll5']].head(6).to_string())

Arsenal 2019-20 first 6 matches — xG_roll5 must lag by 1:
                 date  gameweek        xG  xG_roll5  pts_roll5
0 2019-08-11 14:00:00         1  1.133090       NaN        NaN
1 2019-08-17 12:30:00         2  1.164400       NaN        NaN
2 2019-08-24 17:30:00         3  0.985542       NaN        NaN
3 2019-09-01 16:30:00         4  1.925090  1.094344       2.00
4 2019-09-15 15:30:00         5  1.006160  1.302030       1.75
5 2019-09-22 15:30:00         6  2.532090  1.242856       1.60


## Recency Weights

In [63]:
df['recency_weight'] = df['season'].map(SEASON_WEIGHTS)

print("Recency weight per season:")
print(df.groupby('season')['recency_weight'].first().sort_index())
print(f"\nNull weights: {df['recency_weight'].isna().sum()}")

Recency weight per season:
season
1920    1.0
2021    1.0
2122    1.0
2223    1.1
2324    1.2
2425    1.3
2526    1.5
Name: recency_weight, dtype: float64

Null weights: 0


## Sanity Checks

In [24]:
print("=" * 60)
print("SANITY CHECK 1 — Shape and columns")
print("=" * 60)
print(f"Shape  : {df.shape}")
print(f"Columns: {list(df.columns)}")

SANITY CHECK 1 — Shape and columns
Shape  : (1064, 36)
Columns: ['league', 'season', 'game_id', 'date', 'team', 'opponent', 'venue', 'xG', 'xGA', 'scored', 'conceded', 'result', 'points', 'gd', 'xgd', 'gameweek', 'cum_pts', 'leader_pts', 'title_gap', 'cl_gap', 'eur_gap', 'is_rivalry', 'stakes_intensity', 'is_high_stakes', 'xG_z', 'xGA_z', 'xG_roll5', 'xG_roll10', 'xGA_roll5', 'xGA_roll10', 'pts_roll5', 'pts_roll10', 'is_win', 'win_rate_roll5', 'win_rate_roll10', 'recency_weight']


In [25]:
print("=" * 60)
print("SANITY CHECK 2 — Null counts in critical columns")
print("=" * 60)

critical = ['xG', 'xGA', 'xG_z', 'xGA_z', 'result', 'points',
            'gameweek', 'title_gap', 'cl_gap', 'eur_gap',
            'stakes_intensity', 'recency_weight']
print(df[critical].isna().sum())

SANITY CHECK 2 — Null counts in critical columns
xG                  0
xGA                 0
xG_z                0
xGA_z               0
result              0
points              0
gameweek            0
title_gap           0
cl_gap              0
eur_gap             0
stakes_intensity    0
recency_weight      0
dtype: int64


In [26]:
print("=" * 60)
print("SANITY CHECK 3 — Both Drop Index versions")
print("Expected: Arsenal has highest Drop_Index and Drop_Index_z")
print("=" * 60)

print("RAW (xG units):")
print(drop_index[['drop_xG', 'drop_xGA', 'Drop_Index']].round(3).sort_values('Drop_Index', ascending=False))
print()
print("Z-SCORE (standard deviation units):")
print(drop_index[['drop_xG_z', 'drop_xGA_z', 'Drop_Index_z']].round(3).sort_values('Drop_Index_z', ascending=False))

SANITY CHECK 3 — Both Drop Index versions
Expected: Arsenal has highest Drop_Index and Drop_Index_z
RAW (xG units):
                   drop_xG  drop_xGA  Drop_Index
Manchester United    0.072     0.044       0.058
Arsenal             -0.015     0.086       0.035
Liverpool            0.008    -0.040      -0.016
Manchester City      0.041    -0.112      -0.035

Z-SCORE (standard deviation units):
                   drop_xG_z  drop_xGA_z  Drop_Index_z
Manchester United      0.077       0.050         0.064
Arsenal               -0.015       0.103         0.044
Liverpool              0.007      -0.052        -0.022
Manchester City        0.041      -0.147        -0.053


In [64]:
print("=" * 60)
print("SANITY CHECK 4 — Leakage check")
print("GW1 xG_roll5 must be NaN for all teams")
print("=" * 60)

gw1 = df[df['gameweek'] == 1]
print(f"GW1 xG_roll5 nulls: {gw1['xG_roll5'].isna().sum()} / {len(gw1)}  (should be {len(gw1)})")

SANITY CHECK 4 — Leakage check
GW1 xG_roll5 must be NaN for all teams
GW1 xG_roll5 nulls: 28 / 28  (should be 28)


In [45]:
print("=" * 60)
print("SANITY CHECK 5 — stakes_intensity range and rivalry")
print("Max must be 1.0. Rivalry matches should have visible stakes boost.")
print("=" * 60)

print(f"Min : {df['stakes_intensity'].min():.4f}  (should be >= 0)")
print(f"Max : {df['stakes_intensity'].max():.4f}  (should be 1.0)")
print()
print("Rivalry matches per team:")
print(df.groupby('team')['is_rivalry'].sum())
print()
print("Mean stakes — rivalry vs non-rivalry (rivalry should be higher):")
print(df.groupby('is_rivalry')['stakes_intensity'].mean().round(3))
print()
top = df.loc[df['stakes_intensity'].idxmax()]
print(f"Highest stakes match: {top['team']} vs {top['opponent']}, "
      f"GW{top['gameweek']} {top['season']}, stakes={top['stakes_intensity']}")

SANITY CHECK 5 — stakes_intensity range and rivalry
Max must be 1.0. Rivalry matches should have visible stakes boost.
Min : 0.0000  (should be >= 0)
Max : 1.0000  (should be 1.0)

Rivalry matches per team:
team
Arsenal              42
Liverpool            42
Manchester City      42
Manchester United    56
Name: is_rivalry, dtype: int64

Mean stakes — rivalry vs non-rivalry (rivalry should be higher):
is_rivalry
False    0.344
True     0.483
Name: stakes_intensity, dtype: float64

Highest stakes match: Arsenal vs Manchester City, GW33 2526, stakes=1.0


In [46]:
print("=" * 60)
print("SANITY CHECK 6 — Z-score properties per team")
print("xG_z and xGA_z should have mean ≈ 0, std ≈ 1 per team")
print("=" * 60)

z_check = df.groupby('team')[['xG_z', 'xGA_z']].agg(['mean', 'std']).round(3)
print(z_check)

SANITY CHECK 6 — Z-score properties per team
xG_z and xGA_z should have mean ≈ 0, std ≈ 1 per team
                  xG_z      xGA_z     
                  mean  std  mean  std
team                                  
Arsenal           -0.0  1.0   0.0  1.0
Liverpool         -0.0  1.0   0.0  1.0
Manchester City   -0.0  1.0  -0.0  1.0
Manchester United -0.0  1.0  -0.0  1.0


In [47]:
print("=" * 60)
print("SANITY CHECK 7 — Season distribution (the big picture)")
print("Check: contention seasons have both high/normal buckets.")
print("Check: early bad Arsenal seasons have some rivalry spikes.")
print("=" * 60)

season_dist = (
    df.groupby(['team', 'season', 'is_high_stakes'])
    .agg(
        n          = ('xG',              'count'),
        avg_xG     = ('xG',              'mean'),
        avg_xGA    = ('xGA',             'mean'),
        avg_stakes = ('stakes_intensity','mean'),
        avg_pts    = ('points',          'mean'),
    )
    .round(3)
)
print(season_dist.to_string())

SANITY CHECK 7 — Season distribution (the big picture)
Check: contention seasons have both high/normal buckets.
Check: early bad Arsenal seasons have some rivalry spikes.
                                          n  avg_xG  avg_xGA  avg_stakes  avg_pts
team              season is_high_stakes                                          
Arsenal           1920   False           29   1.355    1.421       0.172    1.448
                         True             9   1.279    1.782       0.472    1.556
                  2021   False           30   1.367    1.051       0.184    1.567
                         True             8   1.406    1.464       0.456    1.750
                  2122   False           22   1.780    1.333       0.158    1.909
                         True            16   1.514    1.191       0.561    1.688
                  2223   False           20   2.102    0.838       0.148    2.500
                         True            18   1.915    1.578       0.701    1.889
         

## Save to Processed

In [65]:
df.to_csv(PROC_DATA_DIR / "all_4teams_processed.csv", index=False)
print(f"✅ all_4teams_processed.csv  : {len(df)} rows, {df.shape[1]} cols")

for team in TITLE_TEAMS:
    team_df  = df[df['team'] == team]
    filename = f"{team.lower().replace(' ', '_')}_processed.csv"
    team_df.to_csv(PROC_DATA_DIR / filename, index=False)
    print(f"✅ {filename:<42} : {len(team_df)} rows")

drop_index.to_csv(PROC_DATA_DIR / "drop_index.csv")
print(f"✅ drop_index.csv  (raw + z-score Drop Index, contention seasons only)")

✅ all_4teams_processed.csv  : 1064 rows, 36 cols
✅ arsenal_processed.csv                      : 266 rows
✅ liverpool_processed.csv                    : 266 rows
✅ manchester_city_processed.csv              : 266 rows
✅ manchester_united_processed.csv            : 266 rows
✅ drop_index.csv  (raw + z-score Drop Index, contention seasons only)
